# Business problem

* a global e-com company operating accross multiple regions manages end-to-end order fulfillment , including shipping aand delivery , for products like sporting goods.

* the company is facing inconsistant delivery performance , where actual shipping time often devite from schedual timelines , leading , to late deliveries and unpredictable order profitability.

Desire outcomes:
* the goal is to anylize delivery operation , identify bottlenecks (kaha pe actual problem a raha hai) , and build predictive system to reduce delay , optimize shipping decision ans imporove overall profitaility and efficiancy.

In [42]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [9]:
df = pd.read_csv('SupplyChainDataset.csv' , encoding='latin-1')
display(df.head(5))

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [10]:
df.columns

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id',
       'Customer Lname', 'Customer Password', 'Customer Segment',
       'Customer State', 'Customer Street', 'Customer Zipcode',
       'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market',
       'Order City', 'Order Country', 'Order Customer Id',
       'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id',
       'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id',
       'Order Item Product Price', 'Order Item Profit Ratio',
       'Order Item Quantity', 'Sales', 'Order Item Total',
       'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
       'Order Zipcode', 'Product Card Id', 'Product Category Id',
       'Product De

In [13]:
print(f"data shape : {df.shape}")
print('-'*40)
print(f"duplicate data : {df.duplicated().sum()}")
print('-'*40)
print(f'null value : \n{df.isnull().sum()}')
print('-'*40)
print(f"data type : \n{df.dtypes}")

data shape : (180519, 53)
----------------------------------------
duplicate data : 0
----------------------------------------
null value : 
Type                                  0
Days for shipping (real)              0
Days for shipment (scheduled)         0
Benefit per order                     0
Sales per customer                    0
Delivery Status                       0
Late_delivery_risk                    0
Category Id                           0
Category Name                         0
Customer City                         0
Customer Country                      0
Customer Email                        0
Customer Fname                        0
Customer Id                           0
Customer Lname                        8
Customer Password                     0
Customer Segment                      0
Customer State                        0
Customer Street                       0
Customer Zipcode                      3
Department Id                         0
Department Name    

In [14]:
drop_columns = ['Product Description','Product Image','Order Zipcode','Category Id','Customer Email',
                'Customer Fname','Customer Id',
                'Customer Lname',
                'Customer Password',
                'Customer Zipcode',
                'Latitude',
                'Longitude',
                'Order Customer Id',
                'Order Id',
                'Order Item Cardprod Id',
                'Order Item Discount',
                'Order Item Discount Rate',
                'Order Item Id',
                'Order Item Quantity',
                'Order Item Total',
                'Order Zipcode',
                'Product Card Id',
                'Product Category Id',
                ]
df = df.drop(columns=drop_columns)

In [ ]:
drop_columns = ['Market','Customer State' , 'Order State','Order City','Order Country','Customer City','Product Status']
df = df.drop(columns=drop_columns)

In [ ]:
# remove cancelld order since they are not relavant in delivery time analysis
df = df[df['Delivery Status'] != 'Shipping canceled'] 
df['Delivery Status'].value_counts()

Delivery Status
Late delivery       98977
Advance shipping    41592
Shipping on time    32196
Name: count, dtype: int64

In [26]:
data_cnvrt = ['shipping date (DateOrders)' , 'order date (DateOrders)']

for c in data_cnvrt:
    df[c] = pd.to_datetime(df[c] , errors='coerce', dayfirst=False)

In [29]:
print(f"after remove cancel order : {df.shape}")
print(f"after null handel : \n{df.isnull().sum()}")

after remove cancel order : (172765, 24)
after null handel : 
Type                             0
Days for shipping (real)         0
Days for shipment (scheduled)    0
Benefit per order                0
Sales per customer               0
Delivery Status                  0
Late_delivery_risk               0
Category Name                    0
Customer Country                 0
Customer Segment                 0
Customer Street                  0
Department Id                    0
Department Name                  0
order date (DateOrders)          0
Order Item Product Price         0
Order Item Profit Ratio          0
Sales                            0
Order Profit Per Order           0
Order Region                     0
Order Status                     0
Product Name                     0
Product Price                    0
shipping date (DateOrders)       0
Shipping Mode                    0
dtype: int64


In [ ]:
df.head(5)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer Country,Customer Segment,...,Order Item Product Price,Order Item Profit Ratio,Sales,Order Profit Per Order,Order Region,Order Status,Product Name,Product Price,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,Sporting Goods,Puerto Rico,Consumer,...,327.75,0.29,327.75,91.250000,Southeast Asia,COMPLETE,Smart watch,327.75,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,Sporting Goods,Puerto Rico,Consumer,...,327.75,-0.80,327.75,-249.089996,South Asia,PENDING,Smart watch,327.75,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,Sporting Goods,EE. UU.,Consumer,...,327.75,-0.80,327.75,-247.779999,South Asia,CLOSED,Smart watch,327.75,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,Sporting Goods,EE. UU.,Home Office,...,327.75,0.08,327.75,22.860001,Oceania,COMPLETE,Smart watch,327.75,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,Sporting Goods,Puerto Rico,Corporate,...,327.75,0.45,327.75,134.210007,Oceania,PENDING_PAYMENT,Smart watch,327.75,2018-01-15 11:24:00,Standard Class


In [ ]:
for col in df.columns:
    if(df[col]).nunique() < 10:  # category < 10
        print(f"\nvalue count : {col}")
        print(df[col].value_counts())


value count : Type
Type
DEBIT       69295
TRANSFER    42129
PAYMENT     41725
CASH        19616
Name: count, dtype: int64

value count : Days for shipping (real)
Days for shipping (real)
2    54205
6    27489
3    27478
4    27297
5    27003
0     4839
1     4454
Name: count, dtype: int64

value count : Days for shipment (scheduled)
Days for shipment (scheduled)
4    103153
2     33806
1     26513
0      9293
Name: count, dtype: int64

value count : Delivery Status
Delivery Status
Late delivery       98977
Advance shipping    41592
Shipping on time    32196
Name: count, dtype: int64

value count : Late_delivery_risk
Late_delivery_risk
1    98977
0    73788
Name: count, dtype: int64

value count : Customer Country
Customer Country
EE. UU.        106425
Puerto Rico     66340
Name: count, dtype: int64

value count : Customer Segment
Customer Segment
Consumer       89420
Corporate      52528
Home Office    30817
Name: count, dtype: int64

value count : Order Status
Order Status
COMPLETE  

In [ ]:
df['Order_process_time'] = (df['shipping date (DateOrders)'] - df['order date (DateOrders)']).dt.days
df['delay'] = df['Order_process_time'] - df['Days for shipment (scheduled)']
df['is_delay'] = df['delay'] >0
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day_name()
df['order_hour'] = df['order date (DateOrders)'].dt.hour

df.describe()

,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Department Id,order date (DateOrders),Order Item Product Price,Order Item Profit Ratio,Sales,Order Profit Per Order,Product Price,shipping date (DateOrders),Order_process_time,delay,order_month,order_hour
count,172765.000000,172765.000000,172765.000000,172765.000000,172765.000000,172765.000000,172765,172765.000000,172765.000000,172765.000000,172765.000000,172765.000000,172765,172765.000000,172765.000000,172765.000000,172765.000000
mean,3.498596,2.933100,22.032360,183.165948,0.572900,5.444095,2016-06-12 15:25:39.457991936,141.278595,0.120801,203.828493,22.032360,141.278595,2016-06-16 03:25:14.452927488,3.472816,0.539716,6.235511,11.482604
min,0.000000,0.000000,-4274.979980,7.490000,0.000000,2.000000,2015-01-01 00:00:00,9.990000,-2.750000,9.990000,-4274.979980,9.990000,2015-01-03 00:00:00,0.000000,-2.000000,1.000000,0.000000
25%,2.000000,2.000000,7.030000,104.379997,0.000000,4.000000,2015-09-21 18:01:00,50.000000,0.080000,119.980003,7.030000,50.000000,2015-09-25 08:59:00,2.000000,0.000000,3.000000,5.000000
50%,3.000000,4.000000,31.520000,163.990005,1.000000,5.000000,2016-06-11 08:11:00,59.990002,0.270000,199.919998,31.520000,59.990002,2016-06-15 03:38:00,3.000000,1.000000,6.000000,11.000000
75%,5.000000,4.000000,64.800003,247.399994,1.000000,7.000000,2017-02-28 21:08:00,199.990005,0.360000,299.950012,64.800003,199.990005,2017-03-04 08:00:00,5.000000,1.000000,9.000000,17.000000
max,6.000000,4.000000,911.799988,1939.989990,1.000000,12.000000,2018-01-31 23:38:00,1999.989990,0.500000,1999.989990,911.799988,1999.989990,2018-02-06 22:14:00,6.000000,4.000000,12.000000,23.000000
std,1.623446,1.373405,104.355313,120.141871,0.494659,1.629248,NaN,139.862956,0.466610,132.392520,104.355313,139.862956,NaN,1.670187,1.494150,3.405593,6.927276


In [43]:
df.to_csv('clean_data.csv' , index=False)